# Setup

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
import numpy as np

# Train-test Split & Normalization
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Baseline Model
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

#Class Imbalance Handling
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# 1. Data Loading & Missing-value Check

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

df = pd.read_csv("Bankruptcy.csv")

print(f" Dataset Shape : {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print("\nClass distribution:")
print(df["Bankrupt?"].value_counts())

 Dataset Shape : (6819, 96)

Columns:
['Bankrupt?', ' ROA(C) before interest and depreciation before interest', ' ROA(A) before interest and % after tax', ' ROA(B) before interest and depreciation after tax', ' Operating Gross Margin', ' Realized Sales Gross Margin', ' Operating Profit Rate', ' Pre-tax net Interest Rate', ' After-tax net Interest Rate', ' Non-industry income and expenditure/revenue', ' Continuous interest rate (after tax)', ' Operating Expense Rate', ' Research and development expense rate', ' Cash flow rate', ' Interest-bearing debt interest rate', ' Tax rate (A)', ' Net Value Per Share (B)', ' Net Value Per Share (A)', ' Net Value Per Share (C)', ' Persistent EPS in the Last Four Seasons', ' Cash Flow Per Share', ' Revenue Per Share (Yuan ¥)', ' Operating Profit Per Share (Yuan ¥)', ' Per Share Net profit before tax (Yuan ¥)', ' Realized Sales Gross Profit Growth Rate', ' Operating Profit Growth Rate', ' After-tax Net Profit Growth Rate', ' Regular Net Profit Growth 

In [ ]:
print("Missing values:", df.isnull().sum().sum())
df.isnull().sum().sort_values(ascending=True)

Missing values: 0


,0
Bankrupt?,0
ROA(C) before interest and depreciation before interest,0
ROA(A) before interest and % after tax,0
ROA(B) before interest and depreciation after tax,0
Operating Gross Margin,0
...,...
Liability to Equity,0
Degree of Financial Leverage (DFL),0
Interest Coverage Ratio (Interest expense to EBIT),0
Net Income Flag,0


## 2. Preprocessing Pipeline

In [ ]:
X = df.drop(columns=["Bankrupt?"])
y = df["Bankrupt?"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape[0])
print("Testing shape:", X_test.shape[0])

# Same class proportion in training & test sets
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True))

Training shape: 5455
Testing shape: 1364

Training class distribution:
Bankrupt?
0    0.967736
1    0.032264
Name: proportion, dtype: float64

Testing class distribution:
Bankrupt?
0    0.967742
1    0.032258
Name: proportion, dtype: float64


# 3. Baseline Model Development

In [ ]:
# 1. Majority Classifier
majority = DummyClassifier(strategy="most_frequent")

majority.fit(X_train, y_train)

# 2. Logistic Regression
logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

logistic.fit(X_train, y_train)

# 3. Shallow Decision Tree
dt_bl = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_bl.fit(X_train, y_train)

DummyClassifier(strategy='most_frequent')

Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression())])

DecisionTreeClassifier(max_depth=5, random_state=42)

In [ ]:
# Define baseline models
baseline_models = {
    "Majority Classifier": majority,
    "Logistic Regression": logistic,
    "Shallow Decision Tree": dt_bl
}

# Define 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1-score": "f1",
    "Average Precision": "average_precision"
}

# Store evaluation results
baseline_results = []

for name, model in baseline_models.items():

    # Evaluate using cross-validation on training data
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    # Calculate mean and standard deviation across folds
    baseline_results.append({
        "Model": name,
        "Accuracy": scores["test_Accuracy"].mean(),
        "Precision": scores["test_Precision"].mean(),
        "Recall": scores["test_Recall"].mean(),
        "F1-score": scores["test_F1-score"].mean(),
        "Average Precision": scores["test_Average Precision"].mean(),
        "AP Std": scores["test_Average Precision"].std()
    })

# Convert results into a DataFrame
baseline_results_df = pd.DataFrame(baseline_results)

display(baseline_results_df.round(4))

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.p

,Model,Accuracy,Precision,Recall,F1-score,Average Precision,AP Std
0,Majority Classifier,0.9677,0.0000,0.0000,0.0000,0.0323,0.0004
1,Logistic Regression,0.9674,0.4849,0.2160,0.2973,0.3560,0.0300
2,Shallow Decision Tree,0.9644,0.4229,0.2387,0.3015,0.2810,0.0683


In [ ]:
for name, model in baseline_models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    print(f"\n{name}")
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()

    print(f"True Negatives: {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Positives: {tp}")



Majority Classifier
Confusion Matrix:
[[1320    0]
 [  44    0]]
True Negatives: 1320
False Positives: 0
False Negatives: 44
True Positives: 0

Logistic Regression
Confusion Matrix:
[[1306   14]
 [  38    6]]
True Negatives: 1306
False Positives: 14
False Negatives: 38
True Positives: 6

Shallow Decision Tree
Confusion Matrix:
[[1303   17]
 [  26   18]]
True Negatives: 1303
False Positives: 17
False Negatives: 26
True Positives: 18


# 4. Model Development

## Decision Tree with Undersampling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Split BEFORE undersampling
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["Bankrupt?"],
    random_state=42
)

# 2. Separate classes in the training set
positive = train_df[train_df["Bankrupt?"] == 1]
negative = train_df[train_df["Bankrupt?"] == 0]

# 3. Choose the negative-to-positive ratio
# 1 = 1:1; 3 = 3:1; 5 = 5:1
ratio = 5

n_negative_to_keep = min(
    len(negative),
    ratio * len(positive)
)

negative_sample = negative.sample(
    n=n_negative_to_keep,
    replace=False,
    random_state=42
)

# 4. Combine sampled positives with ALL negatives, then shuffle
train_subsample = pd.concat(
    [negative_sample, positive],
    ignore_index=True
).sample(frac=1, random_state=42)

# 5. Separate predictors and target
X_train = train_subsample.drop(columns="Bankrupt?")
y_train = train_subsample["Bankrupt?"]

X_test = test_df.drop(columns="Bankrupt?")
y_test = test_df["Bankrupt?"]

# 6. Train
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
model.fit(X_train, y_train)

# 7. Evaluate on the ORIGINAL test distribution
y_pred = model.predict(X_test)

print("Training counts after undersampling:")
print(y_train.value_counts())

print("\nConfusion matrix — label order [0, 1]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))



DecisionTreeClassifier(max_depth=5, random_state=42)

Training counts after undersampling:
Bankrupt?
0    880
1    176
Name: count, dtype: int64

Confusion matrix — label order [0, 1]:
[[1228   92]
 [  20   24]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.93      0.96      1320
           1       0.21      0.55      0.30        44

    accuracy                           0.92      1364
   macro avg       0.60      0.74      0.63      1364
weighted avg       0.96      0.92      0.94      1364



## Tunded Decision Tree using GridSearchCV

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

pipeline = Pipeline([
    ("sampler", RandomUnderSampler(random_state=42)),
    ("model", DecisionTreeClassifier(random_state=42))
])
# df must be the ORIGINAL dataframe, before undersampling
target = "Bankrupt?"

X = df.drop(columns=target)
y = df[target]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Original training counts:")
print(y_train_full.value_counts())

print("\nOriginal training majority:minority ratio:")
counts = y_train_full.value_counts()
print(counts.max() / counts.min())
# For binary RandomUnderSampler:
# sampling_strategy = minority count / retained majority count
# 1/3 -> 3:1 non-bankrupt:bankrupt
# 1/5 -> 5:1
# 0.1 -> 10:1
param_grid = [
    {
        "sampler__sampling_strategy": [1/3, 1/5, 0.1],
        "model__max_depth": [3, 5, 8, None],
        "model__min_samples_leaf": [5, 10, 20],
        "model__class_weight": [None]
    },
    {
        # Compare against using all training observations
        "sampler": ["passthrough"],
        "model__max_depth": [3, 5, 8, None],
        "model__min_samples_leaf": [5, 10, 20],
        "model__class_weight": [None, "balanced"]
    }
]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={
        "ap": "average_precision",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    refit="ap",
    cv=cv,
    n_jobs=-1
)

search.fit(X_train_full, y_train_full)

print("Best parameters:", search.best_params_)
print("Best CV average precision:", search.best_score_)

# Evaluate only after choosing the model and any threshold
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
p_bankrupt = best_model.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print(classification_report(y_test, y_pred, zero_division=0))
print("Test average precision:",
      average_precision_score(y_test, p_bankrupt))


Original training counts:
Bankrupt?
0    5279
1     176
Name: count, dtype: int64

Original training majority:minority ratio:
29.994318181818183


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('sampler',
                                        RandomUnderSampler(random_state=42)),
                                       ('model',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid=[{'model__class_weight': [None],
                          'model__max_depth': [3, 5, 8, None],
                          'model__min_samples_leaf': [5, 10, 20],
                          'sampler__sampling_strategy': [0.3333333333333333,
                                                         0.2, 0.1]},
                         {'model__class_weight': [None, 'balanced'],
                          'model__max_depth': [3, 5, 8, None],
                          'model__min_samples_leaf': [5, 10, 20],
                          'sampler': ['passthrough']}],
             refit='ap',
             scoring={'ap': 'average_precision', 'f1': 'f1',
                      'precision': 'precision', 'recall': 'recall'})

Best parameters: {'model__class_weight': None, 'model__max_depth': 5, 'model__min_samples_leaf': 10, 'sampler__sampling_strategy': 0.1}
Best CV average precision: 0.2958210359646198
[[1262   58]
 [  20   24]]
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      1320
           1       0.29      0.55      0.38        44

    accuracy                           0.94      1364
   macro avg       0.64      0.75      0.68      1364
weighted avg       0.96      0.94      0.95      1364

Test average precision: 0.29806841115109606


## Cost-Sensitive Decision Tree

In [ ]:
import numpy as np

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.metrics import (
    make_scorer,
    confusion_matrix,
    classification_report,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------------
# 1. Define the cost matrix
# Rows = actual labels; columns = predicted labels
# Label order: [0 = not bankrupt, 1 = bankrupt]
# --------------------------------------------------
COST_FP = 1.0
COST_FN = 20.0

cost_matrix = np.array([
    [0.0,     COST_FP],
    [COST_FN, 0.0]
])


def mean_classification_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    total_cost = np.sum(cm * cost_matrix)
    return total_cost / len(y_true)


cost_scorer = make_scorer(
    mean_classification_cost,
    greater_is_better=False
)

# --------------------------------------------------
# 2. Split ORIGINAL data before undersampling
# --------------------------------------------------
target = "Bankrupt?"

X = df.drop(columns=target)
y = df[target]

if set(y.unique()) != {0, 1}:
    raise ValueError("The target must contain labels 0 and 1 only.")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Original training counts:")
print(y_train_full.value_counts())

if y_train_full.value_counts().min() < 5:
    raise ValueError(
        "At least 5 training observations per class are needed "
        "for this 5-fold stratified search."
    )

# --------------------------------------------------
# 3. Define CV and check feasible sampling ratios
# --------------------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store the splits so feasibility checks and search use
# exactly the same folds.
cv_splits = list(cv.split(X_train_full, y_train_full))

minimum_feasible_ratio = max(
    y_train_full.iloc[train_idx].value_counts().min()
    / y_train_full.iloc[train_idx].value_counts().max()
    for train_idx, _ in cv_splits
)

candidate_ratios = [
    1.0,       # 1:1 majority:minority
    1 / 2,     # 2:1
    1 / 3,     # 3:1
    1 / 5,     # 5:1
    0.1        # 10:1
]

valid_ratios = [
    ratio for ratio in candidate_ratios
    if ratio >= minimum_feasible_ratio
]

print("\nFeasible sampling_strategy values:")
print(valid_ratios)

# --------------------------------------------------
# 4. Pipeline and parameter grid
# --------------------------------------------------
pipeline = Pipeline([
    ("sampler", RandomUnderSampler(random_state=42)),
    ("model", DecisionTreeClassifier(random_state=42))
])

param_grid = [
    {
        # All original training observations
        "sampler": ["passthrough"],
        "model__max_depth": [3, 5, 8, None],
        "model__min_samples_leaf": [5, 10, 20],
        "model__class_weight": [
            None,
            "balanced",
            {0: 1, 1: 5},
            {0: 1, 1: 20}
        ]
    }
]

if valid_ratios:
    param_grid.append({
        "sampler": [RandomUnderSampler(random_state=42)],
        "sampler__sampling_strategy": valid_ratios,
        "model__max_depth": [3, 5, 8, None],
        "model__min_samples_leaf": [5, 10, 20],
        "model__class_weight": [None]
    })

# --------------------------------------------------
# 5. Select the model by minimum CV mean cost
# --------------------------------------------------
search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={
        "neg_cost": cost_scorer,
        "ap": "average_precision",
        "precision": make_scorer(
            precision_score, zero_division=0
        ),
        "recall": make_scorer(
            recall_score, zero_division=0
        ),
        "f1": make_scorer(
            f1_score, zero_division=0
        )
    },
    refit="neg_cost",
    cv=cv_splits,
    n_jobs=-1,
    error_score="raise"
)

search.fit(X_train_full, y_train_full)

print("\nBest parameters:")
print(search.best_params_)

print("\nBest CV mean cost per firm:")
print(round(-search.best_score_, 4))

# CV metrics for the model selected by cost
best_idx = search.best_index_

print("\nCV metrics for the selected model:")
for metric in ["precision", "recall", "f1", "ap"]:
    value = search.cv_results_[f"mean_test_{metric}"][best_idx]
    print(f"{metric}: {value:.4f}")

# --------------------------------------------------
# 6. Evaluate selected model on unchanged test set
# --------------------------------------------------
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

bankrupt_column = list(best_model.classes_).index(1)
p_bankrupt = best_model.predict_proba(X_test)[:, bankrupt_column]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

test_total_cost = float(np.sum(cm * cost_matrix))
test_mean_cost = test_total_cost / len(y_test)

print("\nTest confusion matrix — label order [0, 1]:")
print(cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print("\nTest classification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Not bankrupt", "Bankrupt"],
    zero_division=0
))

print("Test average precision:",
      round(average_precision_score(y_test, p_bankrupt), 4))

print("\nCost breakdown:")
print(f"False-positive cost: {fp} × {COST_FP} = {fp * COST_FP:.2f}")
print(f"False-negative cost: {fn} × {COST_FN} = {fn * COST_FN:.2f}")
print(f"Total test cost: {test_total_cost:.2f}")
print(f"Mean test cost per firm: {test_mean_cost:.4f}")

# --------------------------------------------------
# 7. Compare against trivial decision rules
# --------------------------------------------------
always_healthy = np.zeros(len(y_test), dtype=int)
always_bankrupt = np.ones(len(y_test), dtype=int)

print("\nBaseline total test costs:")
for name, predictions in [
    ("Always predict not bankrupt", always_healthy),
    ("Always predict bankrupt", always_bankrupt)
]:
    baseline_cost = (
        mean_classification_cost(y_test, predictions)
        * len(y_test)
    )
    print(f"{name}: {baseline_cost:.2f}")


Original training counts:
Bankrupt?
0    5279
1     176
Name: count, dtype: int64

Feasible sampling_strategy values:
[1.0, 0.5, 0.3333333333333333, 0.2, 0.1]


## Logistic Regression

In [ ]:
# --------------------------------------------------
# 1. Split the ORIGINAL dataset
# --------------------------------------------------
target = "Bankrupt?"

X = df.drop(columns=target)
y = df[target]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Original training counts:")
print(y_train_full.value_counts())

# --------------------------------------------------
# 2. Pipeline
# Preprocessing and sampling happen within each CV fold
# --------------------------------------------------
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("sampler", RandomUnderSampler(
        sampling_strategy=0.3333333333333333,
        random_state=42
    )),
    ("model", LogisticRegression(
        solver="lbfgs",
        max_iter=10000,
        class_weight=None
    ))
])

# Keep sampling fixed at 10:1 to compare against
# your selected decision-tree configuration.
param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# --------------------------------------------------
# 3. Select regularization strength using CV
# --------------------------------------------------
logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring={
        "ap": "average_precision",
        "precision": make_scorer(
            precision_score, zero_division=0
        ),
        "recall": make_scorer(
            recall_score, zero_division=0
        ),
        "f1": make_scorer(
            f1_score, zero_division=0
        )
    },
    refit="ap",
    cv=cv,
    n_jobs=-1,
    error_score="raise"
)

logistic_search.fit(X_train_full, y_train_full)

best_logistic = logistic_search.best_estimator_

print("\nBest parameters:")
print(logistic_search.best_params_)

print("\nBest CV average precision:")
print(round(logistic_search.best_score_, 4))

# --------------------------------------------------
# 4. Show final undersampled training counts
# --------------------------------------------------
retained_indices = (
    best_logistic.named_steps["sampler"].sample_indices_
)

print("\nFinal undersampled training counts:")
print(
    y_train_full.iloc[retained_indices]
    .value_counts()
    .sort_index()
)

# --------------------------------------------------
# 5. Evaluate on unchanged test data
# --------------------------------------------------
y_pred = best_logistic.predict(X_test)

bankrupt_column = list(best_logistic.classes_).index(1)
p_bankrupt = best_logistic.predict_proba(X_test)[:, bankrupt_column]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("\nConfusion matrix — label order [0, 1]:")
print(cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Not bankrupt", "Bankrupt"],
    zero_division=0
))

print(
    "Test average precision:",
    round(average_precision_score(y_test, p_bankrupt), 4)
)

print(
    "Test ROC-AUC:",
    round(roc_auc_score(y_test, p_bankrupt), 4)
)

# --------------------------------------------------
# 6. Report illustrative investment-error cost
# This DOES NOT control model selection above.
# --------------------------------------------------
COST_FP = 1
COST_FN = 20

total_cost = COST_FP * fp + COST_FN * fn

print("\nIllustrative cost evaluation: FN=20, FP=1")
print(f"Total test cost: {total_cost}")
print(f"Mean cost per firm: {total_cost / len(y_test):.4f}")


Original training counts:
Bankrupt?
0    5279
1     176
Name: count, dtype: int64


NameError: name 'SimpleImputer' is not defined

## AdaBoost

In [ ]:
# ==========================================
# AdaBoost (Adapted for Bankruptcy)
# ==========================================
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

# 1. Define the base learner (Decision Stump)
base_stump = DecisionTreeClassifier(max_depth=1, random_state=42)

# 2. Create the pipeline (Undersample -> AdaBoost)
# We use the same undersampling ratio (3:1) you found best in your Decision Tree tuning
adaboost_pipeline = Pipeline([
    ("sampler", RandomUnderSampler(sampling_strategy=1/3, random_state=42)),
    ("model", AdaBoostClassifier(
        estimator=base_stump,
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ))
])

# 3. Train the model
adaboost_pipeline.fit(X_train_full, y_train_full)

# 4. Predict and Evaluate
y_pred_ada = adaboost_pipeline.predict(X_test)
p_bankrupt_ada = adaboost_pipeline.predict_proba(X_test)[:, 1]

print("=== AdaBoost Results ===")
print("Confusion Matrix [0, 1]:")
print(confusion_matrix(y_test, y_pred_ada, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ada, labels=[0, 1], target_names=["Not bankrupt", "Bankrupt"], zero_division=0))

print("\nTest Average Precision (AP):", round(average_precision_score(y_test, p_bankrupt_ada), 4))

## Gradient Boosting

In [ ]:
# ==========================================
# Gradient Boosting (Adapted for Bankruptcy)
# ==========================================
from sklearn.ensemble import GradientBoostingClassifier

# 1. Create the pipeline (Undersample -> Gradient Boosting)
gb_pipeline = Pipeline([
    ("sampler", RandomUnderSampler(sampling_strategy=1/3, random_state=42)),
    ("model", GradientBoostingClassifier(
        loss="log_loss",
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    ))
])

# 2. Train the model
gb_pipeline.fit(X_train_full, y_train_full)

# 3. Predict and Evaluate
y_pred_gb = gb_pipeline.predict(X_test)
p_bankrupt_gb = gb_pipeline.predict_proba(X_test)[:, 1]

print("\n=== Gradient Boosting Results ===")
print("Confusion Matrix [0, 1]:")
print(confusion_matrix(y_test, y_pred_gb, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb, labels=[0, 1], target_names=["Not bankrupt", "Bankrupt"], zero_division=0))

print("\nTest Average Precision (AP):", round(average_precision_score(y_test, p_bankrupt_gb), 4))

## Gradient Bossting with SMOTE

In [ ]:
# ==========================================
# PART G: Oversampling (SMOTE) with Gradient Boosting
# ==========================================
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # <-- Import Imputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix

# 1. Create a pipeline with Imputation -> SMOTE -> Gradient Boosting
smote_gb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # <-- Step 1: Fill NaNs
    ("sampler", SMOTE(random_state=42)),           # <-- Step 2: Oversample
    ("model", GradientBoostingClassifier(          # <-- Step 3: Train model
        loss="log_loss",
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    ))
])

# 2. Train the model
smote_gb_pipeline.fit(X_train_full, y_train_full)

# 3. Predict and Evaluate
y_pred_smote = smote_gb_pipeline.predict(X_test)
p_bankrupt_smote = smote_gb_pipeline.predict_proba(X_test)[:, 1]

print("=== Gradient Boosting Results (SMOTE Oversampling) ===")
print("Confusion Matrix [0, 1]:")
print(confusion_matrix(y_test, y_pred_smote, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote, labels=[0, 1], target_names=["Not bankrupt", "Bankrupt"], zero_division=0))

print("\nTest Average Precision (AP):", round(average_precision_score(y_test, p_bankrupt_smote), 4))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Get probabilities
p_bankrupt_gb = gb_pipeline.predict_proba(X_test)[:, 1]

# Calculate precision and recall for different thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, p_bankrupt_gb)

# Plot it
plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.legend(loc="best")
plt.title("Precision vs Recall Trade-off (Gradient Boosting)")
plt.grid(True)
plt.show()